In [1]:
import sys; sys.path.append("../src")
import pandas as pd, numpy as np, time
from common import clean_name, core_name, clean_addr, addr_numbers

s1 = pd.read_pickle("../work/tr_s1.pkl")
s2 = pd.read_pickle("../work/tr_s2.pkl")
s3 = pd.read_pickle("../work/tr_s3.pkl")
gt = pd.read_pickle("../work/tr_gt.pkl")

# ---- 1. Sample: 50k S1 + their true matches + some "no owner" records ----
s1s = s1.sample(50000, random_state=42)
g = gt[gt.matched_entity_ids != ""].copy()
g["mid"] = g.matched_entity_ids.str.split(",")
all_pairs = g[["source1_entity_id", "mid"]].explode("mid")
true_pairs = all_pairs[all_pairs.source1_entity_id.isin(s1s.entity_id)]
print("True pairs in sample:", len(true_pairs))

other = pd.concat([s2, s3])
matched_any = set(all_pairs.mid)
q_true = other[other.entity_id.isin(set(true_pairs.mid))]
q_none = other[~other.entity_id.isin(matched_any)].sample(len(q_true) // 3, random_state=42)
q = pd.concat([q_true, q_none])
del other
print("S1 records:", len(s1s), "| S2/S3 records to match:", len(q))

# ---- 2. Clean both sides ----
def prep(df):
    df = df.copy()
    df["name"] = [clean_name(x) for x in df.business_name]
    df["core"] = [core_name(x) for x in df["name"]]
    df["addr"] = [clean_addr(a, c) for a, c in zip(df.business_address, df.country)]
    df["nums"] = [addr_numbers(x) for x in df["addr"]]
    return df

t = time.time()
s1c, qc = prep(s1s), prep(q)
print("Cleaning time (s):", round(time.time() - t, 1))

True pairs in sample: 172636
S1 records: 50000 | S2/S3 records to match: 230181
Cleaning time (s): 19.2


In [2]:
import numpy as np, time
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

def tfidf_topk(s1c, qc, col="core", K=50):
    """For each S2/S3 record, find the K most similar S1 names (same country)."""
    out = []
    for c in qc.country.unique():
        a, b = s1c[s1c.country == c], qc[qc.country == c]
        if len(a) == 0:
            continue
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), min_df=2,
                              sublinear_tf=True, dtype=np.float32)
        A = vec.fit_transform(a[col]); B = vec.transform(b[col])
        C = sp_matmul_topn(B, A.T.tocsr(), top_n=K, threshold=0.05, sort=True, n_threads=4).tocsr()
        rows = np.repeat(np.arange(C.shape[0]), np.diff(C.indptr))
        rank = np.arange(C.nnz) - np.repeat(C.indptr[:-1], np.diff(C.indptr))
        out.append(pd.DataFrame({"entity_id": b.entity_id.values[rows],
                                 "s1_id": a.entity_id.values[C.indices],
                                 "sim": C.data, "rank": rank}))
    return pd.concat(out, ignore_index=True)

t = time.time()
cand = tfidf_topk(s1c, qc, K=50)
print("Time (s):", round(time.time() - t, 1))

truth_df = true_pairs.rename(columns={"source1_entity_id": "s1_id", "mid": "entity_id"})
hit = cand.merge(truth_df, on=["entity_id", "s1_id"])
for K in [1, 3, 5, 10, 20, 50]:
    print(f"top-{K:<3} pairs: {(cand['rank'] < K).sum():>10,}   recall: {(hit['rank'] < K).sum()/len(truth_df):.3f}")

Time (s): 25.2
top-1   pairs:    230,135   recall: 0.764
top-3   pairs:    690,388   recall: 0.858
top-5   pairs:  1,150,544   recall: 0.883
top-10  pairs:  2,300,625   recall: 0.906
top-20  pairs:  4,599,804   recall: 0.920
top-50  pairs: 11,493,254   recall: 0.938


In [3]:
# Same method, but on the address
t = time.time()
cand_a = tfidf_topk(s1c, qc, col="addr", K=50)
print("Address time (s):", round(time.time() - t, 1))

hit_a = cand_a.merge(truth_df, on=["entity_id", "s1_id"])
print("\nADDRESS alone:")
for K in [1, 5, 10, 20]:
    print(f"top-{K:<3} recall: {(hit_a['rank'] < K).sum()/len(truth_df):.3f}")

# Combine name top-K + address top-K
print("\nNAME + ADDRESS combined:")
for Kn, Ka in [(10, 10), (20, 10), (20, 20), (50, 20)]:
    u = pd.concat([cand[cand["rank"] < Kn][["entity_id", "s1_id"]],
                   cand_a[cand_a["rank"] < Ka][["entity_id", "s1_id"]]]).drop_duplicates()
    r = u.merge(truth_df, on=["entity_id", "s1_id"])
    print(f"name top-{Kn:<3} + addr top-{Ka:<3} pairs: {len(u):>10,}   recall: {len(r)/len(truth_df):.3f}")

Address time (s): 95.7

ADDRESS alone:
top-1   recall: 0.907
top-5   recall: 0.930
top-10  recall: 0.936
top-20  recall: 0.941

NAME + ADDRESS combined:
name top-10  + addr top-10  pairs:  4,375,750   recall: 0.995
name top-20  + addr top-10  pairs:  6,671,184   recall: 0.995
name top-20  + addr top-20  pairs:  8,891,404   recall: 0.996
name top-50  + addr top-20  pairs: 15,775,420   recall: 0.997


In [2]:
import sys; sys.path.append("../src")
import importlib, blocking; importlib.reload(blocking)
from blocking import generate_candidates

t = time.time()
cands = generate_candidates(s1c, qc, k_name=10, k_addr=10)
print("Time (s):", round(time.time() - t, 1))
print("Pairs:", len(cands))

truth_df = true_pairs.rename(columns={"source1_entity_id": "s1_id", "mid": "entity_id"})
found = cands.merge(truth_df, on=["entity_id", "s1_id"])
print("Recall:", round(len(found) / len(truth_df), 3))
print(cands.head())

Time (s): 58.5
Pairs: 4375779
Recall: 0.995
      entity_id         s1_id  name_sim  name_rank  addr_sim  addr_rank
0  S2-100000662  S1-119701924  0.000000       99.0  0.375659        6.0
1  S2-100000662  S1-122422189  0.207983        8.0  0.000000       99.0
2  S2-100000662   S1-19437332  0.251129        3.0  0.000000       99.0
3  S2-100000662  S1-257260283  0.251129        2.0  0.000000       99.0
4  S2-100000662  S1-352173173  0.229606        5.0  0.000000       99.0


In [6]:
# Bigger sample: 100k S1 (double the size)
s1b = s1.sample(100000, random_state=7)
tp_b = all_pairs[all_pairs.source1_entity_id.isin(s1b.entity_id)]

other = pd.concat([s2, s3])
qb_true = other[other.entity_id.isin(set(tp_b.mid))]
qb_none = other[~other.entity_id.isin(matched_any)].sample(len(qb_true) // 3, random_state=7)
qb = pd.concat([qb_true, qb_none])
del other

s1bc, qbc = prep(s1b), prep(qb)
print("S1:", len(s1bc), "| S2/S3:", len(qbc))

t = time.time()
cb = generate_candidates(s1bc, qbc, k_name=10, k_addr=10)
print("Time (s):", round(time.time() - t, 1))
print("Pairs:", len(cb))

truth_b = tp_b.rename(columns={"source1_entity_id": "s1_id", "mid": "entity_id"})
print("Recall:", round(len(cb.merge(truth_b, on=["entity_id", "s1_id"])) / len(truth_b), 3))

S1: 100000 | S2/S3: 461152
Time (s): 336.3
Pairs: 8774748
Recall: 0.992


In [7]:
%pip install -U rapidfuzz lightgbm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys; sys.path.append("../src")
import importlib, features, evaluate
importlib.reload(features); importlib.reload(evaluate)
from features import build_features
from evaluate import load_truth, split_ids, tune_threshold
import lightgbm as lgb, numpy as np, time

# 1. Features for every candidate pair
t = time.time()
feat, FEATS = build_features(cands, s1c, qc)
print("Features:", len(FEATS), "| pairs:", len(feat), "| time (s):", round(time.time() - t))

# 2. Label each pair: 1 = true match, 0 = not
lab = true_pairs.rename(columns={"mid": "entity_id", "source1_entity_id": "s1_id"}).assign(label=1)
feat = feat.merge(lab, on=["entity_id", "s1_id"], how="left")
feat["label"] = feat["label"].fillna(0).astype(int)

# 3. Practice split: records owned by validation S1 businesses go to validation
truth_all = load_truth(gt)
tr_ids, val_ids = split_ids(truth_all.keys())
owner = dict(zip(true_pairs.mid, true_pairs.source1_entity_id))
rng = np.random.default_rng(0)
q_val = {e for e in qc.entity_id if owner.get(e) in val_ids or (e not in owner and rng.random() < 0.2)}
is_val = feat.entity_id.isin(q_val)

# 4. Train LightGBM on the training part
model = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           n_jobs=-1, verbose=-1)
t = time.time()
model.fit(feat.loc[~is_val, FEATS], feat.loc[~is_val, "label"])
print("Training time (s):", round(time.time() - t))

# 5. Predict on validation, apply one-owner rule, find the best threshold
pv = feat.loc[is_val, ["entity_id", "s1_id"]].rename(columns={"entity_id": "other_id"})
pv["prob"] = model.predict_proba(feat.loc[is_val, FEATS])[:, 1]
eval_ids = [s for s in s1s.entity_id if s in val_ids]
best_t, best_s = tune_threshold(pv, truth_all, eval_ids)

# 6. Which features mattered most?
print(pd.Series(model.feature_importances_, FEATS).sort_values(ascending=False).head(10))

Features: 28 | pairs: 4375779 | time (s): 165
Training time (s): 47
threshold 0.30 -> F0.5 0.9927
threshold 0.35 -> F0.5 0.9927
threshold 0.40 -> F0.5 0.9926
threshold 0.45 -> F0.5 0.9926
threshold 0.50 -> F0.5 0.9925
threshold 0.55 -> F0.5 0.9924
threshold 0.60 -> F0.5 0.9921
threshold 0.65 -> F0.5 0.9918
threshold 0.70 -> F0.5 0.9918
threshold 0.75 -> F0.5 0.9916
threshold 0.80 -> F0.5 0.9910
threshold 0.85 -> F0.5 0.9898
threshold 0.90 -> F0.5 0.9878
threshold 0.95 -> F0.5 0.9760
BEST: 0.35 0.9927
name_jw          1658
name_partial     1601
addr_jac         1589
name_sim         1563
name_sim_gap     1509
name_nospace     1473
name_sort        1436
addr_sort        1340
addr_set         1299
name_len_diff    1251
dtype: int32
